In [1]:
import pandas as pd
from pathlib import Path
import gzip
import shutil
import json
import re

In [1]:
#standardised names
from pathlib import Path
import re

raw_data_dir = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\raw_biotrace_txt")

print("="*60)
print("STANDARDIZING FILENAMES")
print("="*60)

renamed_count = 0

for file_path in sorted(raw_data_dir.glob("*.txt")):
    old_name = file_path.name
    
    # Pattern 1: sub###_##_X_TYPE.txt (already correct)
    match1 = re.match(r"(sub\d{3}_\d{2})_([ec])_(ohnemarker|mit|edatemp|bf)\.txt", old_name)
    
    # Pattern 2: sub###_##_TYPE_X.txt (needs fixing)
    match2 = re.match(r"(sub\d{3}_\d{2})_(ohnemarker|mit|edatemp|bf)_([ec])\.txt", old_name)
    
    if match1:
        # Already correct format
        print(f"✓ {old_name} (already correct)")
    
    elif match2:
        # Need to swap: move condition before type
        base = match2.group(1)
        file_type = match2.group(2)
        condition = match2.group(3)
        
        new_name = f"{base}_{condition}_{file_type}.txt"
        new_path = file_path.parent / new_name
        
        print(f"  {old_name}")
        print(f"    → {new_name}")
        
        file_path.rename(new_path)
        renamed_count += 1
    else:
        print(f"⚠ Unknown pattern: {old_name}")

print(f"\n{'='*60}")
print(f"✔ Standardized {renamed_count} filenames")
print(f"{'='*60}")

STANDARDIZING FILENAMES
✓ sub001_01_c_bf.txt (already correct)
✓ sub001_01_c_edatemp.txt (already correct)
✓ sub001_01_c_mit.txt (already correct)
⚠ Unknown pattern: sub001_01_ohnemarker.txt
✓ sub001_02_e_bf.txt (already correct)
✓ sub001_02_e_edatemp.txt (already correct)
✓ sub001_02_e_mit.txt (already correct)
⚠ Unknown pattern: sub001_02_ohnemarker.txt
✓ sub002_01_e_bf.txt (already correct)
✓ sub002_01_e_edatemp.txt (already correct)
✓ sub002_01_e_mit.txt (already correct)
✓ sub002_01_e_ohnemarker.txt (already correct)
✓ sub002_02_c_bf.txt (already correct)
✓ sub002_02_c_edatemp.txt (already correct)
✓ sub002_02_c_mit.txt (already correct)
✓ sub002_02_c_ohnemarker.txt (already correct)
✓ sub003_01_c_bf.txt (already correct)
✓ sub003_01_c_edatemp.txt (already correct)
✓ sub003_01_c_mit.txt (already correct)
✓ sub003_01_c_ohnemarker.txt (already correct)
✓ sub003_02_e_bf.txt (already correct)
✓ sub003_02_e_edatemp.txt (already correct)
✓ sub003_02_e_mit.txt (already correct)
✓ sub003_

In [2]:
#convert to bids ecg
import pandas as pd
import json
import gzip
import shutil
from pathlib import Path
import re

raw_data_dir = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\raw_biotrace_txt")
bids_root = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\bids_data")

bids_root.mkdir(exist_ok=True)

SAMPLING_RATE = 256

def parse_biotrace_single_column(filepath):
    """Parse BioTrace file with single column of data (ECG, breathing frequency)"""
    values = []
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # Skip header lines
            if any(keyword in line for keyword in ['Klient:', 'Sitzung:', 'Datum:', 'Zeit:', 'Dauer:', 'Abtastrate:', '256 SPS', 'Sensor']):
                continue
            
            try:
                values.append(float(line))
            except ValueError:
                continue
    
    return pd.DataFrame(values)

def parse_biotrace_edatemp(filepath):
    """Parse BioTrace file with EDA and Temperature data"""
    values_eda = []
    values_temp = []
    
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        # Skip header
        for line in f:
            if 'Sensor' in line or '256 SPS' in line:
                break
        
        # Read data
        for line in f:
            line = line.strip()
            if not line:
                continue
            
            parts = line.split('\t')
            try:
                if len(parts) >= 2:
                    values_eda.append(float(parts[0]))
                    values_temp.append(float(parts[1]))
            except ValueError:
                continue
    
    return pd.DataFrame({
        'eda': values_eda,
        'temp': values_temp
    })

print("\n" + "="*60)
print("CONVERTING TO BIDS FORMAT")
print("="*60)

# ============================================
# Process ECG files
# ============================================
print("\n" + "-"*60)
print("PROCESSING ECG FILES")
print("-"*60)

ecg_files = sorted(raw_data_dir.glob("*_ohnemarker.txt"))
print(f"Found {len(ecg_files)} ECG files\n")

for ecg_path in ecg_files:
    match = re.match(r"sub(\d{3})_(\d{2})_([ec])_ohnemarker\.txt", ecg_path.name)
    if not match:
        print(f"⚠ Skipping {ecg_path.name}")
        continue
    
    sub_id = match.group(1)
    ses_id = match.group(2)
    condition = match.group(3)
    
    print(f"sub-{sub_id}_ses-{ses_id} ({'experimental' if condition == 'e' else 'control'})")
    
    try:
        df = parse_biotrace_single_column(ecg_path)
        df.columns = ['cardiac']
        print(f"  → {len(df)} samples ({len(df)/SAMPLING_RATE:.1f} seconds)")
        
        out_dir = bids_root / f"sub-{sub_id}" / f"ses-{ses_id}" / "beh"
        out_dir.mkdir(parents=True, exist_ok=True)
        
        base_name = f"sub-{sub_id}_ses-{ses_id}_task-BBSIG_physio"
        
        # Write TSV (no header)
        tsv_path = out_dir / f"{base_name}.tsv"
        df.to_csv(tsv_path, sep='\t', index=False, header=False)
        
        # Compress
        with open(tsv_path, 'rb') as f_in:
            with gzip.open(str(tsv_path) + ".gz", 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        tsv_path.unlink()
        
        # Write JSON
        json_path = out_dir / f"{base_name}.json"
        metadata = {
            "SamplingFrequency": SAMPLING_RATE,
            "StartTime": 0,
            "Columns": ["cardiac"],
            "Units": ["mV"],
            "Manufacturer": "Mind Media",
            "ManufacturersModelName": "BioTrace+",
            "RecordingType": "continuous",
            "PhysiologicalRecordingType": "ECG",
            "Condition": "experimental" if condition == 'e' else "control",
            "TaskName": "Audiovisual Relaxation",
            "TaskDescription": "Participants were exposed to either constant light and music (control) or stroboscopic light and binaural beats (experimental) to induce relaxation",
            "DeviceSerialNumber": "BioTrace_001"
        }
        
        with open(json_path, 'w') as f:
            json.dump(metadata, f, indent=4)
        
        print(f"  ✓ Created BIDS files\n")
    
    except Exception as e:
        print(f"  ✗ Error: {e}\n")

# ============================================
# Process EDA + Temperature files
# ============================================
print("\n" + "-"*60)
print("PROCESSING EDA + TEMPERATURE FILES")
print("-"*60)

edatemp_files = sorted(raw_data_dir.glob("*_edatemp.txt"))
print(f"Found {len(edatemp_files)} EDA+Temp files\n")

for file_path in edatemp_files:
    match = re.match(r"sub(\d{3})_(\d{2})_([ec])_edatemp\.txt", file_path.name)
    if not match:
        print(f"⚠ Skipping {file_path.name}")
        continue
    
    sub_id = match.group(1)
    ses_id = match.group(2)
    condition = match.group(3)
    
    print(f"sub-{sub_id}_ses-{ses_id} ({'experimental' if condition == 'e' else 'control'})")
    
    try:
        df = parse_biotrace_edatemp(file_path)
        print(f"  → {len(df)} samples ({len(df)/SAMPLING_RATE:.1f} seconds)")
        
        out_dir = bids_root / f"sub-{sub_id}" / f"ses-{ses_id}" / "beh"
        out_dir.mkdir(parents=True, exist_ok=True)
        
        # Save EDA
        base_name_eda = f"sub-{sub_id}_ses-{ses_id}_task-BBSIG_eda"
        tsv_path = out_dir / f"{base_name_eda}.tsv"
        df[['eda']].to_csv(tsv_path, sep='\t', index=False, header=True)
        
        with open(tsv_path, 'rb') as f_in:
            with gzip.open(str(tsv_path) + ".gz", 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        tsv_path.unlink()
        
        json_path = out_dir / f"{base_name_eda}.json"
        metadata_eda = {
            "SamplingFrequency": SAMPLING_RATE,
            "StartTime": 0,
            "Columns": ["eda"],
            "Units": ["µS"],
            "Manufacturer": "Mind Media",
            "ManufacturersModelName": "BioTrace+",
            "RecordingType": "continuous",
            "PhysiologicalRecordingType": "EDA",
            "Condition": "experimental" if condition == 'e' else "control",
            "TaskName": "Audiovisual Relaxation",
            "TaskDescription": "Participants were exposed to either constant light and music (control) or stroboscopic light and binaural beats (experimental) to induce relaxation",
            "DeviceSerialNumber": "BioTrace_001"
        }
        
        with open(json_path, 'w') as f:
            json.dump(metadata_eda, f, indent=4)
        
        print(f"  ✓ Created EDA files")
        
        # Save Temperature
        base_name_temp = f"sub-{sub_id}_ses-{ses_id}_task-BBSIG_temp"
        tsv_path = out_dir / f"{base_name_temp}.tsv"
        df[['temp']].to_csv(tsv_path, sep='\t', index=False, header=True)
        
        with open(tsv_path, 'rb') as f_in:
            with gzip.open(str(tsv_path) + ".gz", 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        tsv_path.unlink()
        
        json_path = out_dir / f"{base_name_temp}.json"
        metadata_temp = {
            "SamplingFrequency": SAMPLING_RATE,
            "StartTime": 0,
            "Columns": ["temp"],
            "Units": ["°C"],
            "Manufacturer": "Mind Media",
            "ManufacturersModelName": "BioTrace+",
            "RecordingType": "continuous",
            "PhysiologicalRecordingType": "Temperature",
            "Condition": "experimental" if condition == 'e' else "control",
            "TaskName": "Audiovisual Relaxation",
            "TaskDescription": "Participants were exposed to either constant light and music (control) or stroboscopic light and binaural beats (experimental) to induce relaxation",
            "DeviceSerialNumber": "BioTrace_001"
        }
        
        with open(json_path, 'w') as f:
            json.dump(metadata_temp, f, indent=4)
        
        print(f"  ✓ Created Temperature files\n")
    
    except Exception as e:
        print(f"  ✗ Error: {e}\n")

# ============================================
# Process Breathing Frequency files (NEW!)
# ============================================
print("\n" + "-"*60)
print("PROCESSING BREATHING FREQUENCY FILES")
print("-"*60)

bf_files = sorted(raw_data_dir.glob("*_bf.txt"))
print(f"Found {len(bf_files)} breathing frequency files\n")

for file_path in bf_files:
    match = re.match(r"sub(\d{3})_(\d{2})_([ec])_bf\.txt", file_path.name)
    if not match:
        print(f"⚠ Skipping {file_path.name}")
        continue
    
    sub_id = match.group(1)
    ses_id = match.group(2)
    condition = match.group(3)
    
    print(f"sub-{sub_id}_ses-{ses_id} ({'experimental' if condition == 'e' else 'control'})")
    
    try:
        df = parse_biotrace_single_column(file_path)
        df.columns = ['respiration']
        print(f"  → {len(df)} samples ({len(df)/SAMPLING_RATE:.1f} seconds)")
        
        out_dir = bids_root / f"sub-{sub_id}" / f"ses-{ses_id}" / "beh"
        out_dir.mkdir(parents=True, exist_ok=True)
        
        base_name_resp = f"sub-{sub_id}_ses-{ses_id}_task-BBSIG_resp"
        tsv_path = out_dir / f"{base_name_resp}.tsv"
        df.to_csv(tsv_path, sep='\t', index=False, header=True)
        
        with open(tsv_path, 'rb') as f_in:
            with gzip.open(str(tsv_path) + ".gz", 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        tsv_path.unlink()
        
        json_path = out_dir / f"{base_name_resp}.json"
        metadata_resp = {
            "SamplingFrequency": SAMPLING_RATE,
            "StartTime": 0,
            "Columns": ["respiration"],
            "Units": ["Hz"],  # Breathing frequency in Hz
            "Manufacturer": "Mind Media",
            "ManufacturersModelName": "BioTrace+",
            "RecordingType": "continuous",
            "PhysiologicalRecordingType": "Respiration",
            "Condition": "experimental" if condition == 'e' else "control",
            "TaskName": "Audiovisual Relaxation",
            "TaskDescription": "Participants were exposed to either constant light and music (control) or stroboscopic light and binaural beats (experimental) to induce relaxation",
            "DeviceSerialNumber": "BioTrace_001"
        }
        
        with open(json_path, 'w') as f:
            json.dump(metadata_resp, f, indent=4)
        
        print(f"  ✓ Created breathing frequency files\n")
    
    except Exception as e:
        print(f"  ✗ Error: {e}\n")

print("\n" + "="*60)
print("✔ ALL DATA CONVERTED TO BIDS FORMAT!")
print("="*60)


CONVERTING TO BIDS FORMAT

------------------------------------------------------------
PROCESSING ECG FILES
------------------------------------------------------------
Found 42 ECG files

⚠ Skipping sub001_01_ohnemarker.txt
⚠ Skipping sub001_02_ohnemarker.txt
sub-002_ses-01 (experimental)
  → 448768 samples (1753.0 seconds)
  ✓ Created BIDS files

sub-002_ses-02 (control)
  → 405261 samples (1583.1 seconds)
  ✓ Created BIDS files

sub-003_ses-01 (control)
  → 435200 samples (1700.0 seconds)
  ✓ Created BIDS files

sub-003_ses-02 (experimental)
  → 404992 samples (1582.0 seconds)
  ✓ Created BIDS files

sub-004_ses-01 (experimental)
  → 451840 samples (1765.0 seconds)
  ✓ Created BIDS files

sub-004_ses-02 (control)
  → 435200 samples (1700.0 seconds)
  ✓ Created BIDS files

sub-005_ses-01 (control)
  → 395520 samples (1545.0 seconds)
  ✓ Created BIDS files

sub-005_ses-02 (experimental)
  → 490240 samples (1915.0 seconds)
  ✓ Created BIDS files

sub-006_ses-01 (experimental)
  → 509

In [2]:
#Add dataset_description.json
from pathlib import Path
import json

bids_root = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\bids_data")

# Create dataset_description.json
dataset_description = {
    "Name": "Stroboscopic Light and Binaural Beats Relaxation Study",
    "BIDSVersion": "1.9.0",
    "DatasetType": "raw",
    "Authors": [
        "Alisa Balik"  # UPDATE THIS
    ],
    "Acknowledgements": "",
    "HowToAcknowledge": "",
    "Funding": [],
    "ReferencesAndLinks": [],
    "DatasetDOI": ""
}

with open(bids_root / "dataset_description.json", 'w') as f:
    json.dump(dataset_description, f, indent=4)

print("✓ Created dataset_description.json")

✓ Created dataset_description.json


In [3]:
#Update JSON sidecars with recommended fields
from pathlib import Path
import json

bids_root = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\bids_data")

print("Updating JSON sidecars with recommended fields...")
print("="*60)

# Find all JSON files
for json_file in bids_root.rglob("*_task-BBSIG_*.json"):
    
    with open(json_file, 'r') as f:
        metadata = json.load(f)
    
    # Add recommended fields
    if '_physio.json' in json_file.name:
        # ECG files
        metadata.update({
            "TaskName": "Audiovisual Relaxation",
            "TaskDescription": "Participants were exposed to either constant light and music (control) or stroboscopic light and binaural beats (experimental) to induce relaxation",
            "InstitutionName": "Freie Universität",  # ADD YOUR INSTITUTION
            "InstitutionAddress": "",
            "DeviceSerialNumber": "BioTrace_001"  # Pseudonym for your device
        })
    elif '_eda.json' in json_file.name or '_temp.json' in json_file.name:
        # EDA and Temperature files
        metadata.update({
            "TaskName": "Audiovisual Relaxation",
            "TaskDescription": "Participants were exposed to either constant light and music (control) or stroboscopic light and binaural beats (experimental) to induce relaxation",
            "InstitutionName": "Freie Universität",  # ADD YOUR INSTITUTION
            "InstitutionAddress": "",
            "DeviceSerialNumber": "BioTrace_001"
        })
    
    # Write updated metadata
    with open(json_file, 'w') as f:
        json.dump(metadata, f, indent=4)
    
    print(f"✓ Updated {json_file.relative_to(bids_root)}")

print("\n✓ All JSON files updated")

Updating JSON sidecars with recommended fields...
✓ Updated sub-001\ses-01\beh\sub-001_ses-01_task-BBSIG_eda.json
✓ Updated sub-001\ses-01\beh\sub-001_ses-01_task-BBSIG_physio.json
✓ Updated sub-001\ses-01\beh\sub-001_ses-01_task-BBSIG_temp.json
✓ Updated sub-001\ses-01\derivatives\preprocessed\sub-001_ses-01_task-BBSIG_physio_preprocessing_info.json
✓ Updated sub-001\ses-02\beh\sub-001_ses-02_task-BBSIG_eda.json
✓ Updated sub-001\ses-02\beh\sub-001_ses-02_task-BBSIG_physio.json
✓ Updated sub-001\ses-02\beh\sub-001_ses-02_task-BBSIG_temp.json
✓ Updated sub-001\ses-02\derivatives\preprocessed\sub-001_ses-02_task-BBSIG_physio_preprocessing_info.json
✓ Updated sub-002\ses-01\beh\sub-002_ses-01_task-BBSIG_eda.json
✓ Updated sub-002\ses-01\beh\sub-002_ses-01_task-BBSIG_physio.json
✓ Updated sub-002\ses-01\beh\sub-002_ses-01_task-BBSIG_temp.json
✓ Updated sub-002\ses-01\derivatives\preprocessed\sub-002_ses-01_task-BBSIG_physio_preprocessing_info.json
✓ Updated sub-002\ses-02\beh\sub-002_ses-

In [6]:
#Fix gzip metadata issue (reproducible compression)
import gzip
import shutil
from pathlib import Path
import os

bids_root = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\bids_data")

print("Fixing gzip metadata...")
print("="*60)

# Find all .gz files
gz_files = list(bids_root.rglob("*.tsv.gz"))
print(f"Found {len(gz_files)} gzipped files\n")

for gz_file in gz_files:
    # Decompress
    tsv_file = gz_file.with_suffix('')
    
    with gzip.open(gz_file, 'rb') as f_in:
        with open(tsv_file, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    
    # Remove old gzip
    gz_file.unlink()
    
    # Recompress with reproducible settings
    # Using command to create gzip without metadata
    with open(tsv_file, 'rb') as f_in:
        with gzip.GzipFile(filename='', mode='wb', fileobj=open(gz_file, 'wb'), mtime=0) as f_out:
            shutil.copyfileobj(f_in, f_out)
    
    # Remove uncompressed file
    tsv_file.unlink()
    
    print(f"✓ Fixed {gz_file.relative_to(bids_root)}")

print(f"\n✓ All gzip files fixed")

Fixing gzip metadata...
Found 122 gzipped files

✓ Fixed sub-001\ses-01\beh\sub-001_ses-01_task-BBSIG_physio.tsv.gz
✓ Fixed sub-001\ses-01\beh\sub-001_ses-01_task-BBSIG_temp.tsv.gz
✓ Fixed sub-001\ses-02\beh\sub-001_ses-02_task-BBSIG_eda.tsv.gz
✓ Fixed sub-001\ses-02\beh\sub-001_ses-02_task-BBSIG_physio.tsv.gz
✓ Fixed sub-001\ses-02\beh\sub-001_ses-02_task-BBSIG_temp.tsv.gz
✓ Fixed sub-002\ses-01\beh\sub-002_ses-01_task-BBSIG_eda.tsv.gz
✓ Fixed sub-002\ses-01\beh\sub-002_ses-01_task-BBSIG_physio.tsv.gz
✓ Fixed sub-002\ses-01\beh\sub-002_ses-01_task-BBSIG_temp.tsv.gz
✓ Fixed sub-002\ses-02\beh\sub-002_ses-02_task-BBSIG_eda.tsv.gz
✓ Fixed sub-002\ses-02\beh\sub-002_ses-02_task-BBSIG_physio.tsv.gz
✓ Fixed sub-002\ses-02\beh\sub-002_ses-02_task-BBSIG_temp.tsv.gz
✓ Fixed sub-003\ses-01\beh\sub-003_ses-01_task-BBSIG_eda.tsv.gz
✓ Fixed sub-003\ses-01\beh\sub-003_ses-01_task-BBSIG_physio.tsv.gz
✓ Fixed sub-003\ses-01\beh\sub-003_ses-01_task-BBSIG_temp.tsv.gz
✓ Fixed sub-003\ses-02\beh\sub-003_

In [2]:
# fix the bids folder structure - derivatives moved
import os
from pathlib import Path

# ── CONFIG ──────────────────────────────────────────────────────────────────
root = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\bids_data")
# ────────────────────────────────────────────────────────────────────────────

moved, failed = [], []

# Walk through every sub-*/ses-*/derivatives/
for deriv_path in sorted(root.glob("sub-*/ses-*/derivatives")):
    if not deriv_path.is_dir():
        continue

    sub = deriv_path.parents[1].name  # e.g. sub-001
    ses = deriv_path.parents[0].name  # e.g. ses-01

    print(f"\nProcessing: {sub}/{ses}/derivatives/")

    # Loop over subfolders: phases, preprocessed, peaks
    for pipeline_folder in sorted(deriv_path.iterdir()):
        if not pipeline_folder.is_dir():
            continue

        # Target: root/derivatives/<subfolder>/sub-XXX/ses-XX/
        target = root / "derivatives" / pipeline_folder.name / sub / ses
        target.mkdir(parents=True, exist_ok=True)

        # Move each file individually
        for item in sorted(pipeline_folder.iterdir()):
            dest = target / item.name

            # Avoid overwriting
            if dest.exists():
                dest = target / (item.stem + "_duplicate" + item.suffix)

            try:
                os.rename(str(item), str(dest))
                moved.append(f"{item.name} → {dest}")
                print(f"  ✅ {item.name} → derivatives/{pipeline_folder.name}/{sub}/{ses}/")
            except Exception as e:
                failed.append(f"{item} — {e}")
                print(f"  ❌ Failed: {item.name} — {e}")

        # Remove the now-empty pipeline subfolder
        try:
            os.rmdir(str(pipeline_folder))
        except Exception as e:
            print(f"  ⚠️  Could not remove {pipeline_folder.name}/: {e}")

    # Remove the now-empty derivatives folder
    try:
        os.rmdir(str(deriv_path))
        print(f"Removed empty: {deriv_path}")
    except Exception as e:
        print(f"⚠️  Could not remove derivatives/: {e} — delete manually if empty")

# ── FINAL REPORT ─────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"✅ Successfully moved: {len(moved)} file(s)")
print(f"❌ Failed:             {len(failed)} file(s)")
if failed:
    print("\nFailed files:")
    for f in failed:
        print(f"   {f}")


Processing: sub-001/ses-01/derivatives/
  ⚠️  Could not remove preprocessed/: [WinError 5] Access is denied: 'C:\\Users\\alisa\\OneDrive\\Desktop\\HRV_Biotrace\\bids_data\\sub-001\\ses-01\\derivatives\\preprocessed'
  ⚠️  Could not remove rpeaks/: [WinError 5] Access is denied: 'C:\\Users\\alisa\\OneDrive\\Desktop\\HRV_Biotrace\\bids_data\\sub-001\\ses-01\\derivatives\\rpeaks'
⚠️  Could not remove derivatives/: [WinError 5] Access is denied: 'C:\\Users\\alisa\\OneDrive\\Desktop\\HRV_Biotrace\\bids_data\\sub-001\\ses-01\\derivatives' — delete manually if empty

Processing: sub-001/ses-02/derivatives/
  ✅ sub-001_ses-02_task-BBSIG_physio_phases.json → derivatives/phases/sub-001/ses-02/
  ⚠️  Could not remove phases/: [WinError 5] Access is denied: 'C:\\Users\\alisa\\OneDrive\\Desktop\\HRV_Biotrace\\bids_data\\sub-001\\ses-02\\derivatives\\phases'
  ✅ sub-001_ses-02_task-BBSIG_physio_preprocessed.tsv → derivatives/preprocessed/sub-001/ses-02/
  ✅ sub-001_ses-02_task-BBSIG_physio_preproce

In [9]:
#txt to tsv conversion eda, temp, bf

import re
import gzip
from pathlib import Path

# =============================================================================
# PATHS -- edit if folder locations differ
# =============================================================================
RAW_DIR   = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\raw_biotrace_txt")
BIDS_ROOT = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\bids_data")
TASK      = "BBSIG"

# =============================================================================
# FILENAME PARSER
# Matches: sub001_01_c_bf.txt  /  sub001_01_e_edatemp.txt
# =============================================================================
FILE_RE = re.compile(r"^sub(\d+)_(\d+)_[ce]_(bf|edatemp)\.txt$", re.IGNORECASE)

def parse_filename(fname):
    m = FILE_RE.match(fname)
    if not m:
        return None
    num, ses, ftype = m.groups()
    return f"sub-{num.zfill(3)}", f"ses-{ses.zfill(2)}", ftype.lower()


# =============================================================================
# READERS
# =============================================================================

def read_bf(path):
    """
    Return list of floats: raw respiratory belt signal.
    Skips all header lines by attempting float conversion on the first field.
    BF data lines contain a single numeric value followed by tabs.
    """
    values = []
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            first = line.strip().split("\t")[0].strip()
            try:
                values.append(float(first))
            except ValueError:
                continue
    return values


def read_edatemp(path):
    """
    Return (eda_list, temp_list).
    Data lines start with hh:mm:ss -- EDA is column 2, TEMP is column 3.
    The timestamp column itself is dropped.
    """
    eda, temp = [], []
    time_re = re.compile(r"^\d{2}:\d{2}:\d{2}")
    with open(path, "r", encoding="latin-1") as f:
        for line in f:
            stripped = line.strip()
            if not stripped or not time_re.match(stripped):
                continue
            parts = stripped.split("\t")
            if len(parts) < 3:
                continue
            try:
                eda.append(float(parts[1].strip()))
                temp.append(float(parts[2].strip()))
            except ValueError:
                continue
    return eda, temp


# =============================================================================
# WRITER
# =============================================================================

def write_gz(path, rows_iter):
    """Write rows to a gzipped TSV with no header, always overwriting."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with gzip.open(path, "wt", encoding="utf-8", newline="\n") as f:
        for row in rows_iter:
            f.write(row + "\n")


# =============================================================================
# MAIN
# =============================================================================

bf_done, et_done, skipped, errors = 0, 0, 0, []

for txt_file in sorted(RAW_DIR.glob("*.txt")):
    parsed = parse_filename(txt_file.name)
    if parsed is None:
        print(f"  skip  {txt_file.name}  (name not recognised)")
        skipped += 1
        continue

    subj_id, ses_id, ftype = parsed
    beh_dir = BIDS_ROOT / subj_id / ses_id / "beh"
    stem    = f"{subj_id}_{ses_id}_task-{TASK}"

    # -- BF -------------------------------------------------------------------
    if ftype == "bf":
        out = beh_dir / f"{stem}_recording-resp_physio.tsv.gz"
        try:
            values = read_bf(txt_file)
            if not values:
                raise ValueError("no numeric data found")
            write_gz(out, (str(v) for v in values))
            print(f"  BF  {subj_id} {ses_id}  {len(values):6d} samples  ->  {out.name}")
            bf_done += 1
        except Exception as e:
            msg = f"  ERR BF  {txt_file.name}: {e}"
            print(msg); errors.append(msg)

    # -- EDA / TEMP -----------------------------------------------------------
    elif ftype == "edatemp":
        out = beh_dir / f"{stem}_recording-edatemp_physio.tsv.gz"
        try:
            eda_vals, temp_vals = read_edatemp(txt_file)
            if not eda_vals:
                raise ValueError("no EDA/TEMP data found")
            if len(eda_vals) != len(temp_vals):
                raise ValueError(
                    f"length mismatch EDA={len(eda_vals)} TEMP={len(temp_vals)}"
                )
            write_gz(out, (f"{e}\t{t}" for e, t in zip(eda_vals, temp_vals)))
            print(f"  ET  {subj_id} {ses_id}  {len(eda_vals):6d} samples  ->  {out.name}")
            et_done += 1
        except Exception as e:
            msg = f"  ERR ET  {txt_file.name}: {e}"
            print(msg); errors.append(msg)

print()
print("=" * 60)
print(f"BF files written    : {bf_done}")
print(f"EDA/TEMP written    : {et_done}")
print(f"Unrecognised names  : {skipped}")
if errors:
    print(f"\n{len(errors)} error(s):")
    for e in errors:
        print(e)
else:
    print("No errors.")
print("=" * 60)


  BF  sub-001 ses-01   57408 samples  ->  sub-001_ses-01_task-BBSIG_recording-resp_physio.tsv.gz
  ET  sub-001 ses-01   57408 samples  ->  sub-001_ses-01_task-BBSIG_recording-edatemp_physio.tsv.gz
  skip  sub001_01_c_mit.txt  (name not recognised)
  skip  sub001_01_ohnemarker.txt  (name not recognised)
  BF  sub-001 ses-02   56224 samples  ->  sub-001_ses-02_task-BBSIG_recording-resp_physio.tsv.gz
  ET  sub-001 ses-02   56224 samples  ->  sub-001_ses-02_task-BBSIG_recording-edatemp_physio.tsv.gz
  skip  sub001_02_e_mit.txt  (name not recognised)
  skip  sub001_02_ohnemarker.txt  (name not recognised)
  BF  sub-002 ses-01   56096 samples  ->  sub-002_ses-01_task-BBSIG_recording-resp_physio.tsv.gz
  ET  sub-002 ses-01   56096 samples  ->  sub-002_ses-01_task-BBSIG_recording-edatemp_physio.tsv.gz
  skip  sub002_01_e_mit.txt  (name not recognised)
  skip  sub002_01_e_ohnemarker.txt  (name not recognised)
  BF  sub-002 ses-02   57792 samples  ->  sub-002_ses-02_task-BBSIG_recording-resp_phy

In [6]:
#bf 18.1 missing file added
"""
Script: Convert sub-018 ses-01 BF raw txt → BIDS + run BF per-phase analysis

Steps:
  1. Parse raw BioTrace txt (32 Hz, respiration belt signal)
  2. Save BIDS-compliant resp TSV.GZ + JSON to bids_data/sub-018/ses-01/beh/
  3. Run NeuroKit2 RSP processing (khodadad2018 method)
  4. Extract per-phase BF_bpm using phases JSON
  5. Patch the 4 new rows into task-BBSIG_per_phase_bf.tsv
     (replaces the existing NaN rows for sub-018 ses-01)

Condition: experimental  (ses-01 for sub-018 is experimental — confirmed from data)
"""

import pandas as pd
import numpy as np
import neurokit2 as nk
import json, gzip, shutil
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ─── PATHS ────────────────────────────────────────────────────────────────────
RAW_FILE  = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\raw_biotrace_txt\sub_018_01_bf.txt")
BIDS_ROOT = Path(r"C:\Users\alisa\OneDrive\Desktop\HRV_Biotrace\bids_data")
BIDS_OUT  = BIDS_ROOT / "sub-018" / "ses-01" / "beh"
BF_TSV    = BIDS_ROOT / "derivatives" / "bf-analysis" / "task-BBSIG_per_phase_bf.tsv"
PHASES_JSON = (BIDS_ROOT / "derivatives" / "phases" /
               "sub-018" / "ses-01" /
               "sub-018_ses-01_task-BBSIG_physio_phases.json")

BIDS_OUT.mkdir(parents=True, exist_ok=True)

SFREQ      = 32          # Hz — confirmed from file header
CONDITION  = "experimental"
SUBJ_ID    = "sub-018"
SES_ID     = "ses-01"
TASK_NAME  = "BBSIG"
PHASE_ORDER = ["pre", "early", "late", "post"]

# ─── STEP 1: PARSE RAW SIGNAL ─────────────────────────────────────────────────
print("=" * 60)
print("STEP 1 — Parsing raw BF signal")
print("=" * 60)

signal_vals = []
with open(RAW_FILE, encoding="latin-1") as f:
    for line in f:
        val = line.split("\t")[0].strip()
        # Skip header lines — only keep numeric values
        try:
            signal_vals.append(float(val))
        except ValueError:
            continue

signal = np.array(signal_vals)
time_s = np.arange(len(signal)) / SFREQ

print(f"  Samples   : {len(signal)}")
print(f"  Duration  : {time_s[-1]:.1f}s ({time_s[-1]/60:.1f} min)")
print(f"  Range     : {signal.min():.1f} – {signal.max():.1f}")

# ─── STEP 2: SAVE BIDS FILES ──────────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2 — Saving BIDS resp TSV.GZ + JSON")
print("=" * 60)

base_name = f"{SUBJ_ID}_{SES_ID}_task-{TASK_NAME}_resp"

# TSV (no header, single column = respiration)
tsv_path = BIDS_OUT / f"{base_name}.tsv"
pd.DataFrame(signal, columns=["respiration"]).to_csv(
    tsv_path, sep="\t", index=False, header=False)

# Compress to TSV.GZ
gz_path = BIDS_OUT / f"{base_name}.tsv.gz"
with open(tsv_path, "rb") as f_in, gzip.open(gz_path, "wb") as f_out:
    shutil.copyfileobj(f_in, f_out)
tsv_path.unlink()  # remove uncompressed version
print(f"  Saved → {gz_path.name}")

# JSON sidecar
json_meta = {
    "SamplingFrequency": SFREQ,
    "StartTime": 0.0,
    "Columns": ["respiration"],
    "Units": "ARU",
    "Manufacturer": "Mind Media",
    "ManufacturersModelName": "BioTrace+",
    "RecordingType": "continuous",
    "PhysiologicalRecordingType": "Respiration",
    "Condition": CONDITION,
    "TaskName": "Audiovisual Relaxation",
    "TaskDescription": (
        "Participants were exposed to either stroboscopic light and "
        "binaural beats (experimental) or constant light and music (control)"
    )
}
json_path = BIDS_OUT / f"{base_name}.json"
with open(json_path, "w") as f:
    json.dump(json_meta, f, indent=4)
print(f"  Saved → {json_path.name}")

# ─── STEP 3: NEUROKIT2 RSP PROCESSING ────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3 — NeuroKit2 RSP processing (khodadad2018)")
print("=" * 60)

rsp_signals, rsp_info = nk.rsp_process(
    signal, sampling_rate=SFREQ, method="khodadad2018")

peaks_idx = rsp_info["RSP_Peaks"]
overall_rate = len(peaks_idx) / (time_s[-1] / 60)
print(f"  Peaks detected : {len(peaks_idx)}")
print(f"  Overall BF     : {overall_rate:.1f} br/min")

# ─── STEP 4: LOAD PHASES + EXTRACT PER-PHASE BF ──────────────────────────────
print("\n" + "=" * 60)
print("STEP 4 — Per-phase BF extraction")
print("=" * 60)

if not PHASES_JSON.exists():
    print(f"  ⚠  Phases JSON not found: {PHASES_JSON}")
    print("     Cannot extract per-phase values.")
    raise FileNotFoundError(PHASES_JSON)

with open(PHASES_JSON) as f:
    phases = json.load(f)["phases"]

new_rows = []
for phase_name in PHASE_ORDER:
    p       = phases[phase_name]
    start_s = p["start"]
    end_s   = p["end"]
    dur_min = (end_s - start_s) / 60

    # Mean instantaneous rate from NeuroKit interpolated signal
    phase_mask        = (time_s >= start_s) & (time_s <= end_s)
    phase_rate_signal = rsp_signals.loc[phase_mask, "RSP_Rate"].values
    bf_bpm            = float(np.nanmean(phase_rate_signal))

    # Mean amplitude
    phase_amp = rsp_signals.loc[phase_mask, "RSP_Amplitude"].values
    mean_amp  = float(np.nanmean(phase_amp))

    # Peaks in phase
    peak_times  = time_s[peaks_idx]
    phase_peaks = peaks_idx[
        (peak_times >= start_s) & (peak_times <= end_s)]

    new_rows.append({
        "subj_id":           SUBJ_ID,
        "ses_id":            SES_ID,
        "condition":         CONDITION,
        "phase":             phase_name,
        "BF_bpm":            round(bf_bpm, 4),
        "BF_mean_amplitude": round(mean_amp, 4),
        "BF_n_cycles":       len(phase_peaks),
        "bf_dur_min":        round(dur_min, 4),
        "bf_excluded":       False
    })

    print(f"  {phase_name.upper():6s}: {bf_bpm:.1f} br/min | "
          f"amp={mean_amp:.1f} | n_peaks={len(phase_peaks)} | "
          f"dur={dur_min:.2f} min")

df_new = pd.DataFrame(new_rows)

# ─── STEP 5: PATCH INTO MAIN BF TSV ──────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 5 — Patching into main BF TSV")
print("=" * 60)

df_bf = pd.read_csv(BF_TSV, sep="\t")
print(f"  Rows before patch: {len(df_bf)}")

# Show what's currently there for sub-018 ses-01
existing = df_bf[
    (df_bf["subj_id"] == SUBJ_ID) &
    (df_bf["ses_id"]  == SES_ID)]
print(f"  Existing sub-018 ses-01 rows: {len(existing)}")
if not existing.empty:
    print(existing[["subj_id","ses_id","condition",
                     "phase","BF_bpm"]].to_string(index=False))

# Remove existing NaN rows for sub-018 ses-01 and replace with new values
df_bf = df_bf[~(
    (df_bf["subj_id"] == SUBJ_ID) &
    (df_bf["ses_id"]  == SES_ID)
)]

# Add new columns if they don't exist yet
for col in ["BF_mean_amplitude", "BF_n_cycles", "bf_dur_min", "bf_excluded"]:
    if col not in df_bf.columns:
        df_bf[col] = np.nan

df_bf = pd.concat([df_bf, df_new], ignore_index=True)

# Sort to keep logical order
df_bf = df_bf.sort_values(["subj_id","ses_id","phase"]).reset_index(drop=True)

df_bf.to_csv(BF_TSV, sep="\t", index=False)
print(f"\n  Rows after patch : {len(df_bf)}")
print(f"\n  New sub-018 ses-01 values:")
print(df_bf[
    (df_bf["subj_id"] == SUBJ_ID) &
    (df_bf["ses_id"]  == SES_ID)
][["subj_id","ses_id","condition","phase","BF_bpm"]].to_string(index=False))

print(f"\n  ✅ Patched BF TSV saved → {BF_TSV.name}")
print("=" * 60)

STEP 1 — Parsing raw BF signal
  Samples   : 58208
  Duration  : 1819.0s (30.3 min)
  Range     : 951.2 – 1177.0

STEP 2 — Saving BIDS resp TSV.GZ + JSON
  Saved → sub-018_ses-01_task-BBSIG_resp.tsv.gz
  Saved → sub-018_ses-01_task-BBSIG_resp.json

STEP 3 — NeuroKit2 RSP processing (khodadad2018)
  Peaks detected : 580
  Overall BF     : 19.1 br/min

STEP 4 — Per-phase BF extraction
  PRE   : 16.0 br/min | amp=27.9 | n_peaks=27 | dur=1.77 min
  EARLY : 19.4 br/min | amp=4.0 | n_peaks=49 | dur=2.50 min
  LATE  : 19.7 br/min | amp=4.6 | n_peaks=59 | dur=3.00 min
  POST  : 19.7 br/min | amp=10.3 | n_peaks=58 | dur=3.00 min

STEP 5 — Patching into main BF TSV
  Rows before patch: 160
  Existing sub-018 ses-01 rows: 4
subj_id ses_id    condition phase  BF_bpm
sub-018 ses-01 experimental early     NaN
sub-018 ses-01 experimental  late     NaN
sub-018 ses-01 experimental  post     NaN
sub-018 ses-01 experimental   pre     NaN

  Rows after patch : 160

  New sub-018 ses-01 values:
subj_id ses